In [8]:
from dataclasses import dataclass
import os
from dotenv import load_dotenv
from openai import OpenAI

import json

load_dotenv()

@dataclass(frozen=True)
class Provider:
    name:str
    env_var: str
    base_url: str
    model: str

PROVIDER = [
    Provider(
        "GROQ_AI",
        "GROQ_AI_KEY",
        "https://api.groq.com/openai/v1",
        "openai/gpt-oss-120b"
    )
]

def select_provider()->Provider:
    for provider in PROVIDER:
        if os.getenv("GROQ_AI_KEY"):
            return provider

    expected=".".join(p.env_var for p in PROVIDER)
    raise RuntimeError(f"No provider key set, add one of the {expected} to your environment variable")

def build_client(provider:Provider)->OpenAI:
    client=OpenAI(api_key=os.getenv(provider.env_var),base_url=provider.base_url)
    return client

def reply_llm(prompt: str):
    newprovider=select_provider()
    newclient=build_client(newprovider)
    result=newclient.chat.completions.create(
        model=newprovider.model,
        messages=[{
            "role":"user",
            "content":prompt
        }]
    )

    return result.choices[0].message.content


WEATHER_DB = {
    "london": {"celcius": 22, "sky": "cloudy"},
    "paris": {"celcius": 24, "sky": "sunny"},
    "tallinn": {"celcius": 28, "sky": "sunny"},
    "moscow": {"celcius": 18, "sky": "rainy"},
    "tokyo": {"celcius": 30, "sky": "sunny"},
    "beijing": {"celcius": 26, "sky": "cloudy"},
    "new york": {"celcius": 28, "sky": "sunny"},
    "mumbai": {"celcius": 32, "sky": "sunny"},
    "cape town": {"celcius": 16, "sky": "cloudy"},
    "sydney": {"celcius": 20, "sky": "sunny"},
    "rio de janeiro": {"celcius": 22, "sky": "cloudy"},
    "cairo": {"celcius": 24, "sky": "sunny"},
    "mexico city": {"celcius": 20, "sky": "cloudy"},
    
}

def look_up(location: str):
    record=WEATHER_DB.get(location.lower().strip())
    if record is None:
        return f"Sorry I don't about a weather at a {location}"

    return f"The weather in {location} is {record['celcius']}C and {record['sky']}."


weather_scehma={
    'type':"function",
    "function":{
        "name":"lookup_weather",
        "description":"Look up weather for a location",
        "parameters":{
            "type":"object",
            "properties":{
                "location":{
                    "type":"string",
                    "description":"The location to look up weather for",
                }
            },
            "required":["location"]
        }

    }
}

Tools={
    "lookup_weather":look_up,
    "stock_price_cheacker":None
}

def ask_llm_with_tool(prompt: str, * ,max_token:int=400)->str:
    """
    Call a llm with tool call
    """
    provider=select_provider()
    client=build_client(provider)

    result=client.chat.completions.create(
        model=provider.model,
        max_tokens=max_token,
        tools=[weather_scehma],
        messages=[{
            "role":"user",
            "content":prompt,
        }]
    )

    print(result.choices)
    return result.choices[0].message


# msg = ask_llm_with_tool("What is the weather in Tokyo?")

# print("final response from llm", msg)


# if msg.tool_calls:
#     for tool_call in msg.tool_calls:
#         city=json.loads(tool_call.function.arguments)["location"]
#         tool=Tools[tool_call.function.name]
#         result =tool(city)
#         print(result)

#         print("-"*100)



def ask__llm(prompt: str,max_token=4000)->str :
    provider=select_provider()
    client=build_client(provider)
    message=[{
                "role":"user",
                "content":prompt,
            }]

    while True:

        result=client.chat.completions.create(
            model=provider.model,
            max_tokens=max_token,
            tools=[weather_scehma],
            messages=message
        )
        msg=result.choices[0].message

        if not msg.tool_calls:
            return msg.content

        message.append({
            "role":"assistant",
            "content":msg.content,
            "tool_calls":msg.tool_calls
        })


        for tool_call in msg.tool_calls:
            tool_name=tool_call.function.name
            tool_arg=json.loads(tool_call.function.arguments)
            too_call_id=tool_call.id

            if tool_name not in Tools:
                raise ValueError(f"tool name not found")
            tool=Tools[tool_name]
            result=tool(**tool_arg)

            message.append({
                "role":"tool",
                "tool_call_id":too_call_id,
                "content":result
            })


ask__llm("What is the weather like in mumbai compared to Paris?")






'Both cities are enjoying sunny weather today. However, Mumbai is considerably warmer—around **32\u202f°C**, while Paris is about **24\u202f°C**. So Mumbai’s temperature is roughly **8\u202f°C higher** than Paris’s, even though the sky conditions are the same in both places.'